# Feasible vs infeasible **selling** — does the 8-quarter continuity mask drive the contemporaneous result?

`pipeline_v2` forbids predicting **sell** for any position lacking a full 8-quarter history:

```python
feas = (hs.min(axis=1) > 0)      # held in ALL 8 quarters
p[~feas[s], 0] = 0.0             # zero the SELL probability
p = p / p.sum(1, keepdims=True)  # mass pushed onto buy/hold
```

The comment calls this feasibility ("can't sell what you don't hold"), but **every ranked position *is* held at t** — it has a rank at t. What the mask actually conditions on is **position age**. That is a much stronger assumption, and it is only harmless if infeasible positions rarely sell and their sells are unremarkable.

**This notebook uses no model.** Everything below is computed directly from the holdings panel and the returns file, so the result does not depend on any LSTM run.

| section | question |
|---|---|
| **D1** | how often do infeasible positions actually **sell**? |
| **D2** | what do sells **earn**, feasible vs infeasible — and is the difference significant? |
| **D3** | how many real sells does the mask **override**? |
| A–C | the weaker version: age → same-quarter co-movement |

One detail that would otherwise invalidate D: **exits are reconstructed**. Held at t, gone at t+1 while the fund still files ⇒ that is a sell, and the raw panel contains no row for it. Uncorrected, sells are undercounted *more among young positions* — faking the very asymmetry under test.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import signals_perf as S          # loaders: ids, daily->quarterly, strict lags
import explore_continuity as E    # the exploration itself

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

# ROOT contains manager_holdings/ and return_data_v2.csv
ROOT = '.'
cfg = E.Config(root=ROOT, seq_len=8, max_rank=25)
cfg

In [ ]:
# Build once and keep the pieces, so later cells can slice without reloading.
df = E.build(cfg)                              # ranked positions + continuity flags
panel = S.load_panels(S.Config(root=ROOT))     # raw panel, needed to spot exits
print(f"\nfeasible (held all {cfg.seq_len}q): {df.feasible.mean():.1%}")
df.head(3)

## D1 — how often do infeasible positions sell?

The mask sets `P(sell) = 0` for these. If their true sell rate is high, it is not a mild correction — it overrides a large fraction of a class.

`P(exit | sell)` is the share of those sells that are full exits, i.e. rows the raw panel does not contain at all.

In [ ]:
sell = E.selling_asymmetry(df, panel)
rates = sell['sell_base_rates']
rates.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
idx = [str(i) for i in rates.index]
w = 0.25
for k, (c, col) in enumerate(zip(['P(sell)', 'P(hold)', 'P(buy)'],
                                 ['#D62728', '#B0B0B0', '#2CA02C'])):
    ax[0].bar(np.arange(len(idx)) + k * w, rates[c], w, label=c, color=col)
ax[0].set_xticks(np.arange(len(idx)) + w); ax[0].set_xticklabels(idx)
ax[0].set_xlabel('feasible (held all 8 quarters)'); ax[0].set_ylabel('probability')
ax[0].set_title('Realised trade direction'); ax[0].legend()

ax[1].bar(idx, rates['P(exit | sell)'], color='#4C78A8')
ax[1].set_xlabel('feasible'); ax[1].set_ylabel('P(exit | sell)')
ax[1].set_title('Share of sells that are full exits')
plt.tight_layout(); plt.show()

if False not in rates.index:
    print('no infeasible rows — check seq_len / panel coverage')
else:
    r = rates.loc[False, 'P(sell)'] / max(rates.loc[True, 'P(sell)'], 1e-9)
    print(f"infeasible positions sell {r:.2f}x as often as feasible ones")

## D2 — what do sells earn, feasible vs infeasible?

Rows with `feasible = None` are the **difference** (infeasible − feasible) as a per-quarter series with its own *t*. That difference is exactly what the mask assumes to be zero.

Read the *windows*, not just the levels: a gap that is large **contemporaneously** and gone by **t+2** is same-quarter co-movement, not information.

In [ ]:
sr = sell['sell_returns']
sr.round(4)

In [ ]:
diff = sr[sr['feasible'].isna()]
fig, ax = plt.subplots(figsize=(8, 4))
cols = ['#D62728' if abs(t) >= 2 else '#B0B0B0' for t in diff['t']]
ax.bar(diff['window'], diff['mean_ret'], color=cols)
for x, (m, t) in enumerate(zip(diff['mean_ret'], diff['t'])):
    ax.text(x, m, f't={t:.2f}', ha='center',
            va='bottom' if m >= 0 else 'top', fontsize=9)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('infeasible − feasible sell return')
ax.set_title('The difference the mask assumes away  (red = |t| ≥ 2)')
plt.tight_layout(); plt.show()

## D3 — how much does the mask actually suppress?

In [ ]:
sell['mask_cost'].round(4).T

## A — return by position age

The weaker mechanism: a young position is one the fund *recently built*, typically into a name that was rising. If so it earns much more **contemporaneously** and no more afterwards.

In [ ]:
ages = E.age_buckets(df)
ages.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ages.plot(kind='bar', ax=ax, width=0.8)
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('quarters held in the last 8'); ax.set_ylabel('mean quarterly return')
ax.set_title('Return by position age — contemporaneous vs forward')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## B — does the age filter change the active-weight spread?

Same sort, same funds; only the eligible row set differs. If the contemporaneous spread depends on including young positions, restricting to continuous ones shrinks it — while the predictive spread should be ~zero either way.

In [ ]:
fe = E.filter_effect(df, cfg)
fe.pivot_table(index='subset', columns='window', values=['spread', 't']).round(4)

## C — continuity as a signal on its own

No active weight involved: continuous minus non-continuous, equal weighted. If it "works" contemporaneously and dies forward, that is the same co-movement in isolation.

In [ ]:
cs = E.continuity_signal(df, cfg)
cs.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
cols = ['#D62728' if abs(t) >= 2 else '#B0B0B0' for t in cs['t']]
ax.bar(cs['window'], cs['spread'], color=cols)
for x, (m, t) in enumerate(zip(cs['spread'], cs['t'])):
    ax.text(x, m, f't={t:.2f}', ha='center',
            va='bottom' if m >= 0 else 'top', fontsize=9)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('continuous − non-continuous'); ax.set_title('Continuity as a standalone signal')
plt.tight_layout(); plt.show()

## How to read this

The mask is doing real work — in the direction you suspect — if:

1. **D1**: infeasible positions sell *more* often than feasible ones (so the mask overrides a large class, not a rare edge case);
2. **D2**: their sells earn a **significantly different contemporaneous** return, and that gap **shrinks or vanishes by t+2**;
3. **B**: the contemporaneous active-weight spread shrinks materially when restricted to continuous positions, while the predictive spread barely moves.

Together that means the mask conditions the sort on **position age**, which is mechanically tied to the *overlapping* return — manufacturing significance in the window that is already contaminated, and nowhere else.

The mask is **not** the explanation if infeasible sells are rare *and* their returns match feasible sells in every window. Then it is a genuine (if oddly-named) feasibility rule and the contemporaneous result comes from somewhere else.

> Note: `Config.feasible_only` in `pipeline_v2` defaults to **`False`**, so a default run applies no sell-gating at all. Check `metrics.json['feasible_only']` before attributing any particular run's numbers to this mask.

In [ ]:
# Save everything.
out = {'age_buckets': ages, 'filter_effect': fe, 'continuity_signal': cs,
       **sell}
for k, v in out.items():
    v.to_csv(f'explore_continuity_{k}.csv')
    print(f'wrote explore_continuity_{k}.csv')